# Data Cleaning Notebook

This notebook cleans and standardizes our BRFSS and NTIA datasets for analysis.

## Goals:
1. Clean BRFSS data (2019, 2021, 2023) - standardize variables, handle missing values, aggregate to state level.
2. Clean NTIA data - extract relevant variables and reshape for merging
3. Create state mapping between datasets
4. Merge datasets on state + year

## Setup

## Research Variables

This analysis examines associations between digital infrastructure and preventive healthcare engagement across U.S. states.

### BRFSS Healthcare Engagement Variables

- **CHECKUP1**: Time since last routine checkup
- **PERSDOC3**: Has a personal healthcare provider
- **FLUSHOT7**: Received flu vaccination
- **EXERANY2**: Physical activity participation
- **CHOLCHK3**: Time since cholesterol screening

### BRFSS Supporting Variables

- **_METSTAT**: Metropolitan status
- **_EDUCAG**: Education level
- **_INCOMG1**: Household income category
- **_STATE**: State identifier
- **_LLCPWT**: Survey weight used for state-level weighted estimates

### NTIA Digital Infrastructure Variables

- **wiredHighSpeedAtHome**: Wired high-speed internet at home
- **homeInternetUser**: Uses internet at home
- **callConfUser**: Uses video/voice calling
- **pcOrTabletUser**: Uses a PC or tablet
- **mobilePhoneUser**: Uses a mobile phone
- **emailUser**: Uses email
- **tooExpensiveMainReason**: Cost reported as the main barrier to internet access

These measures are used to explore whether state-level digital infrastructure is associated with healthcare engagement.

We begin by importing the required libraries and custom project modules.

In [1]:
import os
import sys
sys.path.append('../src')  # Tell Python where to find our custom modules
import pandas as pd
import numpy as np
import config
import data_utils
import cleaning_utils

print("✓ Setup complete")
print(f"Analyzing years: {config.YEARS}")
print(f"BRFSS variables: {config.BRFSS_HEALTHCARE_VARS}")
print(f"NTIA variables: {config.NTIA_VARS}")

✓ Setup complete
Analyzing years: [2019, 2021, 2023]
BRFSS variables: ['CHECKUP1', 'PERSDOC3', 'FLUSHOT7', 'EXERANY2', 'CHOLCHK3']
NTIA variables: ['wiredHighSpeedAtHome', 'homeInternetUser', 'callConfUser', 'pcOrTabletUser', 'mobilePhoneUser', 'emailUser', 'tooExpensiveMainReason']


## Load Raw Datasets

We need to load the three years we'll be studying (2019, 2021, 2023) of the BRFSS data and the NTIA dataset. This will give us the raw survey responses before any cleaning or processing.

In [2]:
# Load BRFSS data for each year we're studying
print("Loading BRFSS data...")
brfss_2019 = data_utils.load_brfss_year(2019)
brfss_2021 = data_utils.load_brfss_year(2021)
brfss_2023 = data_utils.load_brfss_year(2023)

# Load NTIA data, has all years in one file
print("Loading NTIA data...")
ntia_raw = data_utils.load_ntia_data()

# Print the shapes of the loaded datasets (Will take a minute or two due to large size)
print(f"\nBRFSS 2019 shape: {brfss_2019.shape}")
print(f"BRFSS 2021 shape: {brfss_2021.shape}")
print(f"BRFSS 2023 shape: {brfss_2023.shape}")
print(f"NTIA shape: {ntia_raw.shape}")

Loading BRFSS data...
Loading NTIA data...

BRFSS 2019 shape: (418268, 342)
BRFSS 2021 shape: (438693, 303)
BRFSS 2023 shape: (433323, 350)
NTIA shape: (835, 348)


## Identify Variable Name Differences

The BRFSS changes variable names between survey years, so we need to identify which variables are named differently so we can standardize them before combining the years.

In [3]:
# Compare 2023 (our reference year) to other years
print("Comparing 2023 to 2021:")
cleaning_utils.compare_variables(brfss_2023, brfss_2021, 2023, 2021)

print("\n" + "="*50 + "\n")

print("Comparing 2023 to 2019:")
cleaning_utils.compare_variables(brfss_2023, brfss_2019, 2023, 2019)

Comparing 2023 to 2021:

Comparing 2023 Dataset to 2021
Missing columns in 2021 dataset: ['ACTIN13_', 'ACTIN23_', 'ALCDAY4', 'ASPIRIN', 'BPMEDS1', 'CDDISCU1', 'CDHOUS1', 'CDSOCIA1', 'CDWORRY', 'CELLSEX2', 'CELSXBRT', 'CHCOCNC1', 'CHCSCNC1', 'CIMEMLO1', 'CNCRTYP2', 'COVACGE1', 'COVIDACT', 'COVIDNU2', 'COVIDPO1', 'COVIDSM1', 'COVIDVA1', 'CPDEMO1C', 'DIABAGE4', 'DIABEDU1', 'DIABEYE1', 'DIABTYPE', 'DRNKANY6', 'DRNKDRI2', 'DROCDY4_', 'ECIGNOW2', 'EMTSUPRT', 'EXERHMM1', 'EXERHMM2', 'EXEROFT1', 'EXEROFT2', 'EXRACT12', 'EXRACT22', 'FALL12MN', 'FALLINJ5', 'FC601_', 'FEETSORE', 'FIRSTAID', 'FOODSTMP', 'HASYMP1', 'HASYMP2', 'HASYMP3', 'HASYMP4', 'HASYMP5', 'HASYMP6', 'HAVARTH4', 'HEATTBCO', 'IMFVPLA4', 'INDORTAN', 'LANDSEX2', 'LCSCTSC1', 'LCSCTWHN', 'LCSSCNCR', 'LNDSXBRT', 'LSATISFY', 'MARJDAB', 'MARJEAT', 'MARJOTHR', 'MARJSMOK', 'MARJVAPE', 'MAXVO21_', 'MENTCIGS', 'MENTECIG', 'METVL12_', 'METVL22_', 'NUMBURN3', 'NUMHHOL4', 'NUMPHON4', 'PA3MIN_', 'PA3VIGM_', 'PADUR1_', 'PADUR2_', 'PAFREQ1_', 'PAF

## Create Variable Name Standardization Mapping

From the comparison above, we can see several of our key variables have different names across years. For example:
- `PERSDOC3` (2023) appears as `PERSDOC2` in earlier years
- `CHOLCHK3` (2023) was `CHOLCHK2` in 2019
- `_INCOMG1` (2023) was `_INCOMG` in 2019

We'll create dictionaries to rename these variables so all years use consistent 2023 naming conventions.

In [4]:
# Variable name mappings based on comparison results
# 2021 mappings - fewer changes needed
rename_2021 = {
    'PDIABTST': 'PDIABTS1',
    'CDDISCUS': 'CDDISCU1'
}

# 2019 mappings - more extensive changes needed
rename_2019 = {
    'PERSDOC2': 'PERSDOC3',
    'CHOLCHK2': 'CHOLCHK3',
    'HADSIGM3': 'HADSIGM4',
    'LASTSIG3': 'LASTSIG4',
    'PDIABTST': 'PDIABTS1',
    'CDDISCUS': 'CDDISCU1',
    '_INCOMG': '_INCOMG1'
}

print("Variable name mappings created:")
print(f"  2021: {len(rename_2021)} variables to rename")
print(f"  2019: {len(rename_2019)} variables to rename")
print("\nExample 2019 mappings:")
for old, new in list(rename_2019.items())[:3]:
    print(f"  {old} → {new}")

Variable name mappings created:
  2021: 2 variables to rename
  2019: 7 variables to rename

Example 2019 mappings:
  PERSDOC2 → PERSDOC3
  CHOLCHK2 → CHOLCHK3
  HADSIGM3 → HADSIGM4


## Standardize Variable Names

Now we'll apply these mappings to rename the variables in 2019 and 2021 datasets to match 2023 conventions. This is an important step of cleaning that ensures that we can combine data across years without variable name conflicts.

In [5]:
# Rename variables in 2021 and 2019 to match 2023
brfss_2021_renamed = brfss_2021.rename(columns=rename_2021)
brfss_2019_renamed = brfss_2019.rename(columns=rename_2019)

# Verify renaming worked by checking if key variables now exist
print("Verification - checking if key variables now exist in all years:")
print(f"  PERSDOC3 in 2023: {('PERSDOC3' in brfss_2023.columns)}")
print(f"  PERSDOC3 in 2021: {('PERSDOC3' in brfss_2021_renamed.columns)}")
print(f"  PERSDOC3 in 2019: {('PERSDOC3' in brfss_2019_renamed.columns)}")
print(f"\n  _INCOMG1 in 2023: {('_INCOMG1' in brfss_2023.columns)}")
print(f"  _INCOMG1 in 2021: {('_INCOMG1' in brfss_2021_renamed.columns)}")
print(f"  _INCOMG1 in 2019: {('_INCOMG1' in brfss_2019_renamed.columns)}")
print("\n✓ Variable names standardized across all years")

Verification - checking if key variables now exist in all years:
  PERSDOC3 in 2023: True
  PERSDOC3 in 2021: True
  PERSDOC3 in 2019: True

  _INCOMG1 in 2023: True
  _INCOMG1 in 2021: True
  _INCOMG1 in 2019: True

✓ Variable names standardized across all years


## Extract Selected Variables

Now that variable names are standardized, we'll extract only the 9 variables we're studying from each year's dataset. This reduces our data from 300+ columns down to just our research variables, making the data clean and manageable to work with.

In [6]:
# Combine all variables we need
all_brfss_vars = config.BRFSS_HEALTHCARE_VARS + config.BRFSS_CONTROL_VARS + config.BRFSS_ID_VARS + ['_LLCPWT'] # Include weight variable

print(f"Extracting {len(all_brfss_vars)} variables: {all_brfss_vars}")

# Extract only needed columns from each year
brfss_2019_subset = brfss_2019_renamed[all_brfss_vars].copy()
brfss_2021_subset = brfss_2021_renamed[all_brfss_vars].copy()
brfss_2023_subset = brfss_2023[all_brfss_vars].copy()

# Add year column to each dataset for tracking
brfss_2019_subset['YEAR'] = 2019
brfss_2021_subset['YEAR'] = 2021
brfss_2023_subset['YEAR'] = 2023

print(f"\nReduced dataset sizes:")
print(f"  2019: {brfss_2019.shape} → {brfss_2019_subset.shape}")
print(f"  2021: {brfss_2021.shape} → {brfss_2021_subset.shape}")
print(f"  2023: {brfss_2023.shape} → {brfss_2023_subset.shape}")

Extracting 10 variables: ['CHECKUP1', 'PERSDOC3', 'FLUSHOT7', 'EXERANY2', 'CHOLCHK3', '_METSTAT', '_EDUCAG', '_INCOMG1', '_STATE', '_LLCPWT']

Reduced dataset sizes:
  2019: (418268, 342) → (418268, 11)
  2021: (438693, 303) → (438693, 11)
  2023: (433323, 350) → (433323, 11)


## Identify Missing Value Codes

BRFSS uses specific numeric codes for missing responses (7, 9, 77, 99 = Don't know/Refused). Before cleaning, we should check these codes exist in our data (if there are NaN values)

In [7]:
# Check if BRFSS missing codes exist in one of our key variables
print("Checking for BRFSS missing value codes in CHECKUP1 (routine checkup):")
print(brfss_2019_subset['CHECKUP1'].value_counts().sort_index())

print("\n" + "="*50)
print("\nWe can see values like 7 and 9 which represent 'Don't know' and 'Refused'.")
print("These should be NaN (missing) rather than treated as actual responses.")
print("Now converting these codes (7, 9, 77, 99) to NaN across all variables...")

# Apply missing value cleaning
brfss_2019_clean = cleaning_utils.clean_brfss_missing_values(brfss_2019_subset)
brfss_2021_clean = cleaning_utils.clean_brfss_missing_values(brfss_2021_subset)
brfss_2023_clean = cleaning_utils.clean_brfss_missing_values(brfss_2023_subset)

print("\n✓ Missing value codes converted to NaN")
print(f"  2019: {brfss_2019_clean.isna().sum().sum():,} NaN values")
print(f"  2021: {brfss_2021_clean.isna().sum().sum():,} NaN values")
print(f"  2023: {brfss_2023_clean.isna().sum().sum():,} NaN values")

Checking for BRFSS missing value codes in CHECKUP1 (routine checkup):
CHECKUP1
1.0    333733
2.0     38487
3.0     20118
4.0     18658
7.0      4432
8.0      2452
9.0       378
Name: count, dtype: int64


We can see values like 7 and 9 which represent 'Don't know' and 'Refused'.
These should be NaN (missing) rather than treated as actual responses.
Now converting these codes (7, 9, 77, 99) to NaN across all variables...

✓ Missing value codes converted to NaN
  2019: 190,997 NaN values
  2021: 202,950 NaN values
  2023: 202,575 NaN values


## Preview Cleaned Data

At this stage, each row still represents an individual BRFSS respondent. The next steps prepare the data for state-level aggregation and integration with the NTIA measures.

In [8]:
# Display first few rows of cleaned 2019 data
print("Sample of cleaned 2019 BRFSS individual responses:")
print(brfss_2019_clean.head(10))

print("\n" + "="*50)
print("\nData types:")
print(brfss_2019_clean.dtypes)

print("\n" + "="*50)
print("\nMissing values by variable:")
print(brfss_2019_clean.isna().sum())

Sample of cleaned 2019 BRFSS individual responses:
   CHECKUP1  PERSDOC3  FLUSHOT7  EXERANY2  CHOLCHK3  _METSTAT  _EDUCAG  \
0       1.0       1.0       2.0       2.0       2.0       1.0      1.0   
1       1.0       1.0       1.0       1.0       2.0       1.0      3.0   
2       1.0       2.0       1.0       1.0       2.0       1.0      4.0   
3       1.0       1.0       NaN       NaN       2.0       1.0      3.0   
4       1.0       1.0       2.0       2.0       2.0       2.0      3.0   
5       1.0       1.0       NaN       NaN       2.0       1.0      2.0   
6       1.0       1.0       1.0       1.0       2.0       2.0      4.0   
7       1.0       1.0       1.0       1.0       2.0       2.0      4.0   
8       1.0       1.0       1.0       1.0       2.0       2.0      2.0   
9       1.0       1.0       1.0       2.0       2.0       1.0      1.0   

   _INCOMG1  _STATE      _LLCPWT  YEAR  
0       2.0     1.0   135.304080  2019  
1       3.0     1.0  1454.882220  2019  
2       5.0

## Remove Rows with Missing State Identifiers

State identifiers are required for the state-level analysis. Responses without `_STATE` cannot be assigned to a state and are therefore excluded before aggregation.

The number of removed observations is reported below for transparency.

In [9]:
# Check missing state values across all years
print("Missing state identifiers:")
print(f"  2019: {brfss_2019_clean['_STATE'].isna().sum():,} missing ({brfss_2019_clean['_STATE'].isna().sum()/len(brfss_2019_clean)*100:.2f}%)")
print(f"  2021: {brfss_2021_clean['_STATE'].isna().sum():,} missing ({brfss_2021_clean['_STATE'].isna().sum()/len(brfss_2021_clean)*100:.2f}%)")
print(f"  2023: {brfss_2023_clean['_STATE'].isna().sum():,} missing ({brfss_2023_clean['_STATE'].isna().sum()/len(brfss_2023_clean)*100:.2f}%)")

# Remove rows with missing state
brfss_2019_clean = brfss_2019_clean.dropna(subset=['_STATE'])
brfss_2021_clean = brfss_2021_clean.dropna(subset=['_STATE'])
brfss_2023_clean = brfss_2023_clean.dropna(subset=['_STATE'])

print(f"\nRemaining data after removing missing states:")
print(f"  2019: {len(brfss_2019_clean):,} responses")
print(f"  2021: {len(brfss_2021_clean):,} responses")
print(f"  2023: {len(brfss_2023_clean):,} responses")

Missing state identifiers:
  2019: 9,163 missing (2.19%)
  2021: 8,341 missing (1.90%)
  2023: 9,501 missing (2.19%)

Remaining data after removing missing states:
  2019: 409,105 responses
  2021: 430,352 responses
  2023: 423,822 responses


Excellent, we still have over 400,000+ responses for each year in the BRFSS datasets. The data loss isn't as jarring 

## Identify Survey Weight Variable

The BRFSS doesn't survey everyone equally, so we need survey weights to calculate accurate state-level statistics.

For instance, if BRFSS surveyed 100 rural people but only 50 urban people, but the state is actually 70% urban, we need to weight the responses so urban voices count more when calculating the state average.

Different years may use different weight variable names. Let's identify which weight variable is present in our data.

In [10]:
# Check if weight variable is now in our cleaned data
print("Confirming _LLCPWT is in our cleaned datasets:")
print(f"  2019: {'_LLCPWT' in brfss_2019_clean.columns}")
print(f"  2021: {'_LLCPWT' in brfss_2021_clean.columns}")
print(f"  2023: {'_LLCPWT' in brfss_2023_clean.columns}")

if '_LLCPWT' in brfss_2019_clean.columns:
    print("\n✓ Survey weight variable available for aggregation")
    print(f"\nSample weight values from 2019:")
    print(brfss_2019_clean['_LLCPWT'].head())
else:
    print("\n✗ Weight variable missing - need to restart kernel and re-run all cells")

Confirming _LLCPWT is in our cleaned datasets:
  2019: True
  2021: True
  2023: True

✓ Survey weight variable available for aggregation

Sample weight values from 2019:
0     135.304080
1    1454.882220
2     215.576852
3     261.282838
4     535.270103
Name: _LLCPWT, dtype: float64


Great, we can see the weight values are decimal numbers, representing how many people each survey respondent represents in the population

## Aggregate to State-Level Percentages

Now we'll convert individual survey responses into state-level statistics. For each state, we'll calculate what percentage of residents answered "Yes" (code 1) to each healthcare question, using the survey weights to ensure accurate representation.

For example: "In Alabama, 78.5% of residents had a routine checkup in the past year."

In [11]:
def aggregate_to_state(df, year):
    """Aggregate individual responses to weighted state-level percentages"""
    
    state_data = []
    
    for state in df['_STATE'].unique():
        state_df = df[df['_STATE'] == state].copy()
        
        row = {'_STATE': state, 'YEAR': year}
        
        # Calculate weighted percentage for each healthcare variable
        for var in config.BRFSS_HEALTHCARE_VARS:
            # Only count responses of 1 (Yes) vs 2 (No), ignoring other values
            valid_responses = state_df[var].isin([1, 2])
            if valid_responses.sum() > 0:
                yes_responses = state_df[var] == 1
                weighted_pct = (state_df[valid_responses & yes_responses]['_LLCPWT'].sum() / 
                               state_df[valid_responses]['_LLCPWT'].sum() * 100)
                row[f'{var}_pct'] = weighted_pct
            else:
                row[f'{var}_pct'] = None
                
        state_data.append(row)
    
    return pd.DataFrame(state_data)

# Aggregate each year
print("Aggregating to state level...")
brfss_2019_state = aggregate_to_state(brfss_2019_clean, 2019)
brfss_2021_state = aggregate_to_state(brfss_2021_clean, 2021)
brfss_2023_state = aggregate_to_state(brfss_2023_clean, 2023)

print(f"\n2019 state-level data: {brfss_2019_state.shape}")
print(f"2021 state-level data: {brfss_2021_state.shape}")
print(f"2023 state-level data: {brfss_2023_state.shape}")
print("\nSample of 2019 state-level data:")
print(brfss_2019_state.head())

Aggregating to state level...

2019 state-level data: (51, 7)
2021 state-level data: (52, 7)
2023 state-level data: (51, 7)

Sample of 2019 state-level data:
   _STATE  YEAR  CHECKUP1_pct  PERSDOC3_pct  FLUSHOT7_pct  EXERANY2_pct  \
0     1.0  2019     88.992649     84.503681     42.110105     68.509288   
1     2.0  2019     84.432718     86.485489     37.404860     78.318683   
2     4.0  2019     86.371703     91.832503     39.609999     75.933467   
3     5.0  2019     90.143892     92.558563     42.089533     68.765783   
4     6.0  2019     83.203733     87.662391     41.406529     77.586258   

   CHOLCHK3_pct  
0      8.495179  
1     17.271597  
2     11.039914  
3     13.999197  
4     10.473238  


Great, we've successfully converted 400,000+ individual responses per year into ~51 state-level rows with weighted percentages. We're seeing 51 or 52 for the states in the datasets because Washington D.C counts as one, along with a U.S teritory (Puerto Rico or Guam for instance). This won't be a problem, we'll only study states in both BRFSS and NTIA datasets.

## Combine Years and Save State-Level BRFSS Data

Now we'll combine all three years into a single dataset and save it for later analysis.

In [12]:
# Combine all years
brfss_state_all_years = pd.concat([brfss_2019_state, brfss_2021_state, brfss_2023_state], ignore_index=True)

print(f"Combined BRFSS state-level data: {brfss_state_all_years.shape}")
print(f"Years included: {sorted(brfss_state_all_years['YEAR'].unique())}")
print(f"Number of unique states: {brfss_state_all_years['_STATE'].nunique()}")

# Preview the data
print("\nSample of combined data:")
print(brfss_state_all_years.head(10))

# Save to processed folder
output_path = '../data/processed/brfss_state_level.csv'
brfss_state_all_years.to_csv(output_path, index=False)
print(f"\n✓ Saved to: {output_path}")

Combined BRFSS state-level data: (154, 7)
Years included: [2019, 2021, 2023]
Number of unique states: 53

Sample of combined data:
   _STATE  YEAR  CHECKUP1_pct  PERSDOC3_pct  FLUSHOT7_pct  EXERANY2_pct  \
0     1.0  2019     88.992649     84.503681     42.110105     68.509288   
1     2.0  2019     84.432718     86.485489     37.404860     78.318683   
2     4.0  2019     86.371703     91.832503     39.609999     75.933467   
3     5.0  2019     90.143892     92.558563     42.089533     68.765783   
4     6.0  2019     83.203733     87.662391     41.406529     77.586258   
5     8.0  2019     83.037797     89.702631     46.413353     81.322121   
6    10.0  2019     90.104923     87.948642     43.780369     73.378165   
7    11.0  2019     85.146768     89.468512     48.382162     81.018261   
8    12.0  2019     89.170983     88.197207     36.899679     73.522380   
9    13.0  2019     88.948109     85.136507     36.204520     72.142215   

   CHOLCHK3_pct  
0      8.495179  
1     1

Great, our BRFSS state-level data is ready with 154 observations (53 unique states across 3 years). Let's now move onto the NTIA datset.

## Clean NTIA Data - Filter Variables and Years

Now we'll clean the NTIA dataset. The NTIA data is structured differently from BRFSS:
- Rows represent different survey questions (variables)
- Columns represent states and demographics
- We need to extract only our selected digital infrastructure variables for 2019, 2021, 2023

First, let's examine the structure and filter to our variables of interest.

In [13]:
# Examine NTIA structure
print("NTIA dataset structure:")
print(f"Shape: {ntia_raw.shape}")
print(f"\nFirst few rows:")
print(ntia_raw.head())
print(f"\nColumn names (first 10):")
print(ntia_raw.columns[:10].tolist())
print(f"\nUnique values in 'variable' column (first 10):")
print(ntia_raw['variable'].unique()[:10])

NTIA dataset structure:
Shape: (835, 348)

First few rows:
    dataset        variable  \
0  Nov 1994   isHouseholder   
1  Nov 1994        isPerson   
2  Nov 1994  computerAtHome   
3  Nov 1994         isAdult   
4  Oct 1997   isHouseholder   

                                         description       universe    usProp  \
0  Household Reference Person in Universe: Non-In...            NaN  1.000000   
1  Person in Universe: Ages 3+ Not Active-Duty Mi...            NaN  1.000000   
2  Anyone in Household Uses a Desktop, Laptop, or...  isHouseholder  0.242794   
3                                    Person Ages 15+       isPerson  0.810185   
4  Household Reference Person in Universe: Non-In...            NaN  1.000000   

   usPropSE    usCount  usCountSE  age314Prop  age314PropSE  ...  WVCount  \
0  0.000000   99708018     233531         NaN           NaN  ...   752405   
1  0.000000  248509799     723058         1.0           0.0  ...  1770883   
2  0.002097   24208471     216845   

## Filter NTIA to Our Variables and Years

From the preview above, we can see the NTIA data structure:
- The **'variable'** column contains survey question names like 'internetUser', 'computerAtHome'
- The **'dataset'** column shows survey dates like "Nov 1994", "Oct 1997"
- State data is spread across columns (ALProp, AKProp, AZProp, etc.)

We'll need to filter to:
1. Only rows where 'variable' matches our 8 digital infrastructure variables
2. Only years containing 2019, 2021, or 2023 in the dataset column

In [14]:
# Check what years are available in the dataset
print("Available survey years in NTIA:")
print(ntia_raw['dataset'].unique())

print("\n" + "="*50)

# Filter to our selected variables
ntia_filtered = ntia_raw[ntia_raw['variable'].isin(config.NTIA_VARS)].copy()

print(f"\nFiltered to our {len(config.NTIA_VARS)} variables:")
print(f"Original NTIA shape: {ntia_raw.shape}")
print(f"Filtered NTIA shape: {ntia_filtered.shape}")
print(f"\nVariables included:")
print(ntia_filtered['variable'].unique())

Available survey years in NTIA:
['Nov 1994' 'Oct 1997' 'Dec 1998' 'Aug 2000' 'Sep 2001' 'Oct 2003'
 'Oct 2007' 'Oct 2009' 'Oct 2010' 'Jul 2011' 'Oct 2012' 'Jul 2013'
 'Jul 2015' 'Nov 2017' 'Nov 2019' 'Nov 2021' 'Nov 2023']


Filtered to our 7 variables:
Original NTIA shape: (835, 348)
Filtered NTIA shape: (125, 348)

Variables included:
['homeInternetUser' 'wiredHighSpeedAtHome' 'tooExpensiveMainReason'
 'pcOrTabletUser' 'mobilePhoneUser' 'emailUser' 'callConfUser']


Interesting! We extracted 5 variables instead of our planned 8 because the other three (metroYesProp, edCollegeGradProp, income100pProp) aren't survey questions - they represent demographic subgroups. 

We'll use the 5 core digital infrastructure variables we have, since we already have the  demographic controls in our BRFSS data.


## Filter to Analysis Years (2019, 2021, 2023)

Now from the available years of the output above, we can see NTIA includes "Nov 2019", "Nov 2021", and "Nov 2023" - perfect matches for our BRFSS years. Let's examine it more

In [15]:
# Filter to our three analysis years
years_to_keep = ['Nov 2019', 'Nov 2021', 'Nov 2023']
ntia_filtered = ntia_filtered[ntia_filtered['dataset'].isin(years_to_keep)].copy()

print(f"Filtered to analysis years:")
print(f"Shape: {ntia_filtered.shape}")
print(f"Years included: {ntia_filtered['dataset'].unique()}")
print(f"Variables included: {ntia_filtered['variable'].unique()}")

print("\nSample of filtered NTIA data:")
print(ntia_filtered.head())

Filtered to analysis years:
Shape: (39, 348)
Years included: ['Nov 2019' 'Nov 2021' 'Nov 2023']
Variables included: ['callConfUser' 'emailUser' 'wiredHighSpeedAtHome' 'pcOrTabletUser'
 'mobilePhoneUser' 'homeInternetUser' 'tooExpensiveMainReason']

Sample of filtered NTIA data:
      dataset              variable  \
583  Nov 2019          callConfUser   
591  Nov 2019             emailUser   
603  Nov 2019  wiredHighSpeedAtHome   
614  Nov 2019  wiredHighSpeedAtHome   
616  Nov 2019        pcOrTabletUser   

                                           description           universe  \
583  Participates in Online Video or Voice Calls or...  adultInternetUser   
591                                         Uses Email  adultInternetUser   
603     Wired High-Speed Internet Service Used at Home   internetAnywhere   
614     Wired High-Speed Internet Service Used at Home     internetAtHome   
616         Uses a Desktop, Laptop, or Tablet Computer            isAdult   

       usProp  usPropSE

Interesting, some NTIA variables appear multiple times per year because they're calculated for different populations (universes). For example, "wiredHighSpeedAtHome" can be measured as a % of all internet users or % of all households.

We need to select one universe definition per variable for consistency.

In [16]:
# Check which variables have multiple universes
print("Variables with multiple universe definitions:")
for var in ntia_filtered['variable'].unique():
    var_data = ntia_filtered[ntia_filtered['variable'] == var]
    universes = var_data['universe'].unique()
    if len(universes) > 1:
        print(f"  {var}: {len(universes)} universes - {universes}")
    else:
        print(f"  {var}: 1 universe - {universes[0]}")

Variables with multiple universe definitions:
  callConfUser: 1 universe - adultInternetUser
  emailUser: 1 universe - adultInternetUser
  wiredHighSpeedAtHome: 3 universes - ['internetAnywhere' 'internetAtHome' 'isHouseholder']
  pcOrTabletUser: 2 universes - ['isAdult' 'isPerson']
  mobilePhoneUser: 2 universes - ['isAdult' 'isPerson']
  homeInternetUser: 2 universes - ['isAdult' 'isPerson']
  tooExpensiveMainReason: 2 universes - ['isHouseholder' 'noInternetAtHome']


Okay so we see that the variables with multiple options are:

- wiredHighSpeedAtHome
- pcOrTabletUser
- mobilePhoneUser
- homeInternetUser
- tooExpensiveMainReason

We need to select one universe per variable to avoid duplicates.

In [17]:
print("\n" + "="*50)
print("\nSelecting one universe per variable for consistency:")

# Define which universe to keep for each variable with multiple options
universe_selection = {
    'wiredHighSpeedAtHome': 'isHouseholder',  # Household-level infrastructure 
    'pcOrTabletUser': 'isAdult',              # Adult population 
    'mobilePhoneUser': 'isAdult',             # Adult population
    'homeInternetUser': 'isAdult',            # Adult population  
    'tooExpensiveMainReason': 'isHouseholder' # Household-level barrier
}

# Filter to selected universes
for var, universe in universe_selection.items():
    mask = (ntia_filtered['variable'] == var) & (ntia_filtered['universe'] == universe)
    ntia_filtered = ntia_filtered[mask | (ntia_filtered['variable'] != var)]

print(f"\nAfter selecting universes:")
print(f"Shape: {ntia_filtered.shape}")
print(f"Expected: 21 rows (7 variables × 3 years)")



Selecting one universe per variable for consistency:

After selecting universes:
Shape: (21, 348)
Expected: 21 rows (7 variables × 3 years)


Great, now we have 21 rows for 3 years which matches out 7 variables x 3 years.

## Reshape NTIA Data Structure

Currently, NTIA data has states spread across columns (ALProp, AKProp, AZProp...). We need to reshape this into a long format where each row represents one state-variable-year combination, so it matches our BRFSS structure.

We'll also extract only the proportion columns (ending in "Prop") and ignore standard error columns (ending in "SE").

In [18]:
# Identify state proportion columns (all columns ending with "Prop" that are state abbreviations)
state_cols = [col for col in ntia_filtered.columns if col.endswith('Prop') and len(col) == 6]

print(f"Found {len(state_cols)} state proportion columns")
print(f"Sample state columns: {state_cols[:5]}")

# Extract relevant columns for reshaping
ntia_reshape = ntia_filtered[['dataset', 'variable'] + state_cols].copy()

print(f"\nData ready for reshaping:")
print(f"Shape: {ntia_reshape.shape}")
print(ntia_reshape.head())

Found 52 state proportion columns
Sample state columns: ['usProp', 'ALProp', 'AKProp', 'AZProp', 'ARProp']

Data ready for reshaping:
Shape: (21, 54)
      dataset          variable    usProp    ALProp    AKProp    AZProp  \
583  Nov 2019      callConfUser  0.507654  0.437618  0.476446  0.537108   
591  Nov 2019         emailUser  0.902662  0.884714  0.879290  0.933339   
616  Nov 2019    pcOrTabletUser  0.666184  0.616907  0.590651  0.706685   
618  Nov 2019   mobilePhoneUser  0.755674  0.746849  0.706474  0.802573   
624  Nov 2019  homeInternetUser  0.763471  0.761154  0.744373  0.795641   

       ARProp    CAProp    COProp    CTProp  ...    SDProp    TNProp  \
583  0.500060  0.531960  0.557903  0.553616  ...  0.503631  0.472201   
591  0.891156  0.905189  0.952504  0.917821  ...  0.910664  0.858602   
616  0.636694  0.661063  0.742628  0.670962  ...  0.716373  0.638996   
618  0.765992  0.764695  0.814740  0.731902  ...  0.742164  0.713094   
624  0.774163  0.751678  0.843252  0.74

Looks good, we have 21 rows (variables) and state daata spread acorss 52 columns (states) as expected

## Reshape from Wide to Long Format

Now for the creating a structure where each row represents one state-variable-year combination, we'll use pandas melt() to convert state columns into rows.

In [19]:
# Reshape from wide to long format
ntia_long = ntia_reshape.melt(
    id_vars=['dataset', 'variable'],
    value_vars=state_cols,
    var_name='state_col',
    value_name='proportion'
)

# Extract state abbreviation from column name (remove 'Prop' suffix)
ntia_long['state_abbr'] = ntia_long['state_col'].str.replace('Prop', '')

# Convert dataset to year
ntia_long['YEAR'] = ntia_long['dataset'].str.extract(r'(\d{4})').astype(int)

# Drop unnecessary columns
ntia_long = ntia_long[['state_abbr', 'YEAR', 'variable', 'proportion']].copy()

print(f"Reshaped NTIA data:")
print(f"Shape: {ntia_long.shape}")
print(f"Expected: ~1092 rows (52 states × 7 variables × 3 years)")
print(f"\nSample of reshaped data:")
print(ntia_long.head(10))

Reshaped NTIA data:
Shape: (1092, 4)
Expected: ~1092 rows (52 states × 7 variables × 3 years)

Sample of reshaped data:
  state_abbr  YEAR                variable  proportion
0         us  2019            callConfUser    0.507654
1         us  2019               emailUser    0.902662
2         us  2019          pcOrTabletUser    0.666184
3         us  2019         mobilePhoneUser    0.755674
4         us  2019        homeInternetUser    0.763471
5         us  2019  tooExpensiveMainReason    0.037741
6         us  2019    wiredHighSpeedAtHome    0.684249
7         us  2021            callConfUser    0.655967
8         us  2021               emailUser    0.918179
9         us  2021          pcOrTabletUser    0.678022


The output looks good EXCEPT row 0 is showing "us", which indicates the national average, not state. We don't need that for state-level analysis.

Let's remove it, then pivot so each variable becomes its own column, making it ready to merge with our BRFSS data.

In [20]:
# Remove national (us) data, keep only actual states
ntia_long = ntia_long[ntia_long['state_abbr'] != 'us'].copy()

print(f"After removing national data: {ntia_long.shape[0]} rows")

# Pivot so each variable becomes a column
ntia_wide = ntia_long.pivot_table(
    index=['state_abbr', 'YEAR'],
    columns='variable',
    values='proportion'
).reset_index()

# Flatten column names
ntia_wide.columns.name = None

print(f"\nPivoted NTIA data:")
print(f"Shape: {ntia_wide.shape}")
print(f"Expected: ~153 rows (51 states × 3 years)")
print(f"\nColumns: {ntia_wide.columns.tolist()}")
print(f"\nSample:")
print(ntia_wide.head())

After removing national data: 1071 rows

Pivoted NTIA data:
Shape: (153, 9)
Expected: ~153 rows (51 states × 3 years)

Columns: ['state_abbr', 'YEAR', 'callConfUser', 'emailUser', 'homeInternetUser', 'mobilePhoneUser', 'pcOrTabletUser', 'tooExpensiveMainReason', 'wiredHighSpeedAtHome']

Sample:
  state_abbr  YEAR  callConfUser  emailUser  homeInternetUser  \
0         AK  2019      0.476446   0.879290          0.744373   
1         AK  2021      0.679123   0.921977          0.816253   
2         AK  2023      0.689531   0.935302          0.760415   
3         AL  2019      0.437618   0.884714          0.761154   
4         AL  2021      0.598645   0.900282          0.740922   

   mobilePhoneUser  pcOrTabletUser  tooExpensiveMainReason  \
0         0.706474        0.590651                0.031253   
1         0.793094        0.714104                0.036400   
2         0.757568        0.650115                0.024556   
3         0.746849        0.616907                0.047836   
4  

## Create State Identifier Mapping and Joining

Now we can begin our merging. One problem though: BRFSS uses numeric FIPS codes (_STATE) while NTIA uses state abbreviations (state_abbr). We need to create a mapping to join them.

For example: Alabama is _STATE = 1.0 in BRFSS and state_abbr = "AL" in NTIA.

In [21]:
# Create FIPS to state abbreviation mapping
state_mapping = {
    1: 'AL', 2: 'AK', 4: 'AZ', 5: 'AR', 6: 'CA', 8: 'CO', 9: 'CT', 10: 'DE', 11: 'DC',
    12: 'FL', 13: 'GA', 15: 'HI', 16: 'ID', 17: 'IL', 18: 'IN', 19: 'IA', 20: 'KS',
    21: 'KY', 22: 'LA', 23: 'ME', 24: 'MD', 25: 'MA', 26: 'MI', 27: 'MN', 28: 'MS',
    29: 'MO', 30: 'MT', 31: 'NE', 32: 'NV', 33: 'NH', 34: 'NJ', 35: 'NM', 36: 'NY',
    37: 'NC', 38: 'ND', 39: 'OH', 40: 'OK', 41: 'OR', 42: 'PA', 44: 'RI', 45: 'SC',
    46: 'SD', 47: 'TN', 48: 'TX', 49: 'UT', 50: 'VT', 51: 'VA', 53: 'WA', 54: 'WV',
    55: 'WI', 56: 'WY', 66: 'GU', 72: 'PR', 78: 'VI'
}

# Add state_abbr to BRFSS data
brfss_state_all_years['state_abbr'] = brfss_state_all_years['_STATE'].map(state_mapping)

print(f"BRFSS states with abbreviations:")
print(f"Total rows: {len(brfss_state_all_years)}")
print(f"Missing state_abbr: {brfss_state_all_years['state_abbr'].isna().sum()}")
print(f"\nSample:")
print(brfss_state_all_years[['_STATE', 'state_abbr', 'YEAR']].head())

BRFSS states with abbreviations:
Total rows: 154
Missing state_abbr: 0

Sample:
   _STATE state_abbr  YEAR
0     1.0         AL  2019
1     2.0         AK  2019
2     4.0         AZ  2019
3     5.0         AR  2019
4     6.0         CA  2019


Now that all 154 BRFSS rows have state abbreviations for joining, we can join our state-level BRFSS healthcare data with NTIA digital infrastructure data using state_abbr and YEAR as matching keys. 

Only states present in both datasets will be included in the final merged data.

In [22]:
# Merge datasets on state and year
merged_data = brfss_state_all_years.merge(
    ntia_wide,
    on=['state_abbr', 'YEAR'],
    how='inner'  # Only keep states present in both datasets
)

print(f"Merged dataset:")
print(f"Shape: {merged_data.shape}")
print(f"BRFSS rows before merge: {len(brfss_state_all_years)}")
print(f"NTIA rows before merge: {len(ntia_wide)}")
print(f"Merged rows: {len(merged_data)}")
print(f"Rows lost: {len(brfss_state_all_years) - len(merged_data)}")

print(f"\nColumns in merged data:")
print(merged_data.columns.tolist())

print(f"\nSample of merged data:")
print(merged_data.head())

Merged dataset:
Shape: (146, 15)
BRFSS rows before merge: 154
NTIA rows before merge: 153
Merged rows: 146
Rows lost: 8

Columns in merged data:
['_STATE', 'YEAR', 'CHECKUP1_pct', 'PERSDOC3_pct', 'FLUSHOT7_pct', 'EXERANY2_pct', 'CHOLCHK3_pct', 'state_abbr', 'callConfUser', 'emailUser', 'homeInternetUser', 'mobilePhoneUser', 'pcOrTabletUser', 'tooExpensiveMainReason', 'wiredHighSpeedAtHome']

Sample of merged data:
   _STATE  YEAR  CHECKUP1_pct  PERSDOC3_pct  FLUSHOT7_pct  EXERANY2_pct  \
0     1.0  2019     88.992649     84.503681     42.110105     68.509288   
1     2.0  2019     84.432718     86.485489     37.404860     78.318683   
2     4.0  2019     86.371703     91.832503     39.609999     75.933467   
3     5.0  2019     90.143892     92.558563     42.089533     68.765783   
4     6.0  2019     83.203733     87.662391     41.406529     77.586258   

   CHOLCHK3_pct state_abbr  callConfUser  emailUser  homeInternetUser  \
0      8.495179         AL      0.437618   0.884714       

Great, the merge was successful. We lost 8 rows since the state-year combinations existed in BRFSS but not in NTIA (likely the territories like Puerto Rico and Guam we mentioned before).

We have 146 observations across 51 states and 3 years with both healthcare and digital infrastructure data.


## Save Final Merged Dataset

Our cleaning is complete. We now have a dataset with state-level healthcare engagement percentages matched with digital infrastructure measures, ready for analysis.

In [23]:
# Check for any missing values in the merged data
print("Missing values in merged dataset:")
print(merged_data.isna().sum())

# Save to processed folder
output_path = '../data/processed/merged_brfss_ntia.csv'
merged_data.to_csv(output_path, index=False)

print(f"\n✓ Final merged dataset saved to: {output_path}")
print(f"\nFinal dataset summary:")
print(f"  Total observations: {len(merged_data)}")
print(f"  States: {merged_data['state_abbr'].nunique()}")
print(f"  Years: {sorted(merged_data['YEAR'].unique())}")
print(f"  Healthcare variables: {len([c for c in merged_data.columns if c.endswith('_pct')])}")
print(f"  Digital infrastructure variables: {len(config.NTIA_VARS)}")

Missing values in merged dataset:
_STATE                    0
YEAR                      0
CHECKUP1_pct              0
PERSDOC3_pct              0
FLUSHOT7_pct              0
EXERANY2_pct              0
CHOLCHK3_pct              0
state_abbr                0
callConfUser              0
emailUser                 0
homeInternetUser          0
mobilePhoneUser           0
pcOrTabletUser            0
tooExpensiveMainReason    0
wiredHighSpeedAtHome      0
dtype: int64

✓ Final merged dataset saved to: ../data/processed/merged_brfss_ntia.csv

Final dataset summary:
  Total observations: 146
  States: 50
  Years: [2019, 2021, 2023]
  Healthcare variables: 5
  Digital infrastructure variables: 7
